# Etapa 5: Transformers para análise de sentimentos

**Tarefa:** a mesma das etapas 2 e 4, classificar avaliações em **positivo, neutro ou negativo**, com o mesmo conjunto de teste do RePro. Depois, os mesmos modelos são testados nos **comentários do YouTube** rotulados por nós.

**O que muda em relação à etapa 4.** Na rede neural da etapa 4, cada palavra tem um vetor fixo: "bom" é sempre o mesmo vetor. Num Transformer, o vetor de cada palavra é calculado olhando a frase inteira (*atenção*), então "não é bom" e "é bom" ficam bem diferentes já na representação.

**Três jeitos de usar um Transformer**, do mais barato ao mais caro:

| Modelo | O que fazemos | O que é treinado | Pergunta que responde |
|---|---|---|---|
| **XLM-R de tweets** (`cardiffnlp/twitter-xlm-roberta-base-sentiment`) | Usamos pronto, sem treino (*zero-shot*) | nada | Um modelo de rede social, multilíngue, já resolve? |
| **BERTimbau congelado** (`neuralmind/bert-base-portuguese-cased`) | Extraímos um vetor por texto e treinamos uma Regressão Logística em cima | só a Regressão Logística | A representação pré-treinada em português basta? |
| **BERTimbau ajustado** (*fine-tuning*) | Ajustamos o modelo inteiro no RePro, 3 sementes | os 110 milhões de parâmetros | Quanto se ganha ajustando tudo, e a que custo? |

Também registramos o **custo computacional** (tempo de treino, parâmetros e tempo de inferência por texto), porque é ele que decide qual modelo vai para o app.

### Como rodar no Google Colab
1. Abra o notebook no Colab: em [colab.research.google.com](https://colab.research.google.com), aba **GitHub**, cole a URL do repositório e escolha `src-pln/notebooks/05_transformers.ipynb`.
2. Ative a GPU: **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU**.
3. **Ambiente de execução → Executar tudo.** Logo no início, o notebook pede o upload do arquivo `amostra_rotulagem.csv` (no seu computador, em `src-pln/coleta_youtube/dados/`). O arquivo fica só na sessão do Colab; os textos do YouTube não vão para o GitHub.
4. Deixe a aba aberta até o fim (o Colab gratuito derruba sessões ociosas). A seção 4 mostra a previsão de tempo do *fine-tuning*.
5. No fim: **Arquivo → Fazer download → Baixar o .ipynb** (salve por cima deste arquivo, em `src-pln/notebooks/`) e baixe o `resultados_etapa5.zip`, cujos CSVs vão para `src-pln/reports/`.

## 0. Configuração

No Colab, a célula abaixo clona o repositório (que traz o RePro, o código comum de `src/` e o gabarito do YouTube). Localmente, usa a pasta do projeto.

`RAPIDO = True` troca os modelos por versões minúsculas e usa poucos exemplos: serve só para testar o código, os números não significam nada.

In [ ]:
import sys, time, random, copy, io, zipfile
from pathlib import Path

RAPIDO = False          # True = teste do código com modelos minúsculos (resultados sem valor)
NO_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/PedroPintor/social-listening-yt.git"

if NO_COLAB:
    if not Path("/content/social-listening-yt").exists():
        !git clone --depth 1 {REPO} /content/social-listening-yt
    RAIZ = Path("/content/social-listening-yt/src-pln")
else:
    RAIZ = Path("..").resolve()     # rodando a partir de src-pln/notebooks/
sys.path.insert(0, str(RAIZ))
print("Pasta do projeto:", RAIZ)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from src.dados import carregar_repro, dividir_treino_teste, dividir_validacao, CLASSES, SEED

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()     # barras de progresso poluem o notebook salvo
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = DEVICE.type == "cuda"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | dispositivo: {DEVICE}")
if GPU:
    print("GPU:", torch.cuda.get_device_name(0))
elif NO_COLAB:
    print("ATENÇÃO: a GPU não está ativada. Ambiente de execução → Alterar o tipo de ambiente → T4 GPU, "
          "e rode de novo. Na CPU, o fine-tuning leva horas.")

MODELO_ZS = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MODELO_BERT = "neuralmind/bert-base-portuguese-cased"
if RAPIDO:
    MODELO_ZS = MODELO_BERT = "hf-internal-testing/tiny-random-BertModel"

REPORTS = RAIZ / "reports"
# no modo RAPIDO, os CSVs vão para uma pasta temporária, para não sobrescrever resultados reais
SAIDA = REPORTS if not RAPIDO else Path(__import__("tempfile").gettempdir()) / "etapa5_teste"
SAIDA.mkdir(exist_ok=True)

## 1. Dados

**RePro.** Mesmas avaliações, mesmo teste (20%) e mesma validação (10% do treino) das etapas 2 e 4, vindos de `src/dados.py`. Uma diferença: os Transformers recebem o texto **original** (título + ". " + avaliação), sem a limpeza da etapa 2. Eles foram pré-treinados com maiúsculas, números e pontuação, e tirar isso só atrapalharia. As linhas e a divisão são as mesmas, então a comparação continua justa.

**YouTube.** O gabarito são os itens em que os dois rotuladores **concordaram** (`rotulo_final` em `rotulos_consolidados.csv`, versionado sem texto). Os textos vêm da `amostra_rotulagem.csv`, enviada por upload. O notebook só imprime contagens e métricas, nunca os textos.

In [ ]:
dados, _ = carregar_repro()
dados["texto_bruto"] = (dados["review_title"].fillna("") + ". " + dados["review_text"].fillna("")).str.strip()

X_train, X_test, y_train, y_test = dividir_treino_teste(dados)
X_tr, X_val, y_tr, y_val = dividir_validacao(X_train, y_train)

T_tr, T_val, T_te = (dados.loc[x.index, "texto_bruto"].tolist() for x in (X_tr, X_val, X_test))
IDX = {c: i for i, c in enumerate(CLASSES)}
Y_tr, Y_val, Y_te = (np.array([IDX[c] for c in y]) for y in (y_tr, y_val, y_test))

if RAPIDO:
    T_tr, Y_tr, T_val, Y_val, T_te, Y_te = T_tr[:64], Y_tr[:64], T_val[:32], Y_val[:32], T_te[:64], Y_te[:64]

print(f"treino {len(T_tr)} | validação {len(T_val)} | teste {len(T_te)}")
print("Classes no teste:", dict(zip(CLASSES, np.bincount(Y_te, minlength=3))))

In [ ]:
ARQ_AMOSTRA = RAIZ / "coleta_youtube" / "dados" / "amostra_rotulagem.csv"
ARQ_GABARITO = RAIZ / "coleta_youtube" / "rotulos" / "rotulos_consolidados.csv"

if NO_COLAB and not ARQ_AMOSTRA.exists():
    from google.colab import files
    print("Envie o arquivo amostra_rotulagem.csv (no seu computador: src-pln/coleta_youtube/dados/)")
    enviados = files.upload()
    if enviados:
        ARQ_AMOSTRA.parent.mkdir(parents=True, exist_ok=True)
        ARQ_AMOSTRA.write_bytes(next(iter(enviados.values())))

YT = None
if ARQ_AMOSTRA.exists():
    amostra = pd.read_csv(ARQ_AMOSTRA, encoding="utf-8-sig", dtype=str)
    gabarito = pd.read_csv(ARQ_GABARITO, encoding="utf-8-sig", dtype=str)
    gabarito = gabarito[gabarito["rotulo_final"].notna()]          # itens em que os dois concordaram
    YT = gabarito[["id", "tipo", "rotulo_final"]].merge(amostra[["id", "texto"]], on="id")
    print(f"YouTube: {len(YT)} itens com gabarito (de {len(amostra)} rotulados)")
    print(pd.crosstab(YT["tipo"], YT["rotulo_final"], margins=True))
else:
    print("Sem a amostra do YouTube: a avaliação no YouTube (seção 5) será pulada.")

## 2. Funções comuns

- **Métricas:** acurácia, F1 macro e F1 por classe, como nas etapas anteriores.
- **Inferência por texto:** tempo médio para classificar **um** texto por vez (como um app faria), medido no dispositivo do treino e também na **CPU**, que é onde o app roda. Na etapa 4, as latências foram medidas na CPU local; as da CPU do Colab são da mesma ordem, mas não idênticas.
- **Comprimento máximo:** 128 *tokens*. No RePro, 95% das avaliações têm até 65 palavras; a célula abaixo mostra quantas passam do limite.

In [ ]:
MAX_LEN = 128

def metricas(y_true, y_pred):
    """y_true e y_pred como índices (0=positivo, 1=neutro, 2=negativo)."""
    f1s = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"acuracia": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0),
            **{f"f1_{c}": f for c, f in zip(CLASSES, f1s)}}

def fixar_semente(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if GPU:
        torch.cuda.manual_seed_all(s)

def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

def medir_latencia(prever_um, textos, n=50):
    """Milissegundos por texto, classificando um de cada vez."""
    amostra = textos[:n]
    prever_um(amostra[0])                       # aquecimento
    t = time.perf_counter()
    for txt in amostra:
        prever_um(txt)
    return (time.perf_counter() - t) / len(amostra) * 1000

def lotes(lista, tamanho):
    for i in range(0, len(lista), tamanho):
        yield lista[i:i + tamanho]

tok_bert = AutoTokenizer.from_pretrained(MODELO_BERT)
n_tokens = [len(ids) for ids in tok_bert(T_te)["input_ids"]]
print(f"Tokens por texto no teste (BERTimbau): mediana {np.median(n_tokens):.0f}, "
      f"95% até {np.percentile(n_tokens, 95):.0f}; acima de {MAX_LEN}: {np.mean(np.array(n_tokens) > MAX_LEN):.1%}")

## 3. Modelos sem *fine-tuning*

### 3.1 XLM-R de tweets, sem treino (*zero-shot*)
O modelo da Cardiff NLP foi treinado para sentimento em tweets de 8 línguas, incluindo o português. Ele já sai com as três classes (*negative, neutral, positive*), então basta traduzir os rótulos. Nada é treinado com o RePro: é o teste mais direto de "dá para usar um modelo pronto?".

In [ ]:
@torch.no_grad()
def prever_logits(modelo, tok, textos, batch=64, dispositivo=DEVICE):
    saida = []
    for lote in lotes(textos, batch):
        enc = tok(lote, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dispositivo)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dispositivo.type == "cuda"):
            saida.append(modelo(**enc).logits.float().cpu())
    return torch.cat(saida).numpy()

t0 = time.perf_counter()
tok_zs = AutoTokenizer.from_pretrained(MODELO_ZS)
mod_zs = AutoModelForSequenceClassification.from_pretrained(MODELO_ZS, **({"num_labels": 3} if RAPIDO else {}))
mod_zs.to(DEVICE).eval()

# rótulos do modelo (negative/neutral/positive) -> nossos índices
TRADUZ = {"negative": "negativo", "neutral": "neutro", "positive": "positivo"}
nomes = [mod_zs.config.id2label[i].lower() for i in range(mod_zs.config.num_labels)]
MAPA_ZS = np.array([IDX[TRADUZ[n]] if n in TRADUZ else i for i, n in enumerate(nomes)])
print("Rótulos do modelo:", nomes, "->", [CLASSES[i] for i in MAPA_ZS])

def prever_zs(textos, dispositivo=DEVICE):
    return MAPA_ZS[prever_logits(mod_zs, tok_zs, textos, dispositivo=dispositivo).argmax(1)]

pred_zs = prever_zs(T_te)
tempo_zs = time.perf_counter() - t0
lat_zs = medir_latencia(lambda t: prever_zs([t]), T_te)
mod_zs.to("cpu"); lat_zs_cpu = medir_latencia(lambda t: prever_zs([t], torch.device("cpu")), T_te); mod_zs.to(DEVICE)

RES = {}   # nome -> dicionário de resultados no teste do RePro
RES["XLM-R tweets (zero-shot)"] = {**metricas(Y_te, pred_zs), "param_total": n_parametros(mod_zs),
                                   "param_treinaveis": 0, "tempo_treino_s": 0.0,
                                   "inferencia_ms": lat_zs, "inferencia_cpu_ms": lat_zs_cpu}
pd.Series(RES["XLM-R tweets (zero-shot)"]).round(4)

### 3.2 BERTimbau congelado + Regressão Logística
O BERTimbau não é ajustado: cada texto vira um vetor de 768 dimensões (média dos vetores dos *tokens* da última camada) e uma Regressão Logística aprende a separar as classes, como o TF-IDF da etapa 2, só que com uma representação contextual. O `C` da regressão é escolhido na validação, e `class_weight="balanced"` compensa o desbalanceamento, como na etapa 2. O tempo de treino inclui a extração dos vetores do treino e da validação.

In [ ]:
@torch.no_grad()
def vetores(modelo, textos, batch=64, dispositivo=DEVICE):
    saida = []
    for lote in lotes(textos, batch):
        enc = tok_bert(lote, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dispositivo)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dispositivo.type == "cuda"):
            h = modelo(**enc).last_hidden_state.float()
        m = enc["attention_mask"].unsqueeze(-1).float()
        saida.append(((h * m).sum(1) / m.sum(1)).cpu())          # média dos tokens (sem o padding)
    return torch.cat(saida).numpy()

bert_cong = AutoModel.from_pretrained(MODELO_BERT).to(DEVICE).eval()
t0 = time.perf_counter()
V_tr, V_val = vetores(bert_cong, T_tr), vetores(bert_cong, T_val)

melhor = None
for C in [0.1, 1.0, 10.0]:
    rl = LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=SEED).fit(V_tr, Y_tr)
    f1v = f1_score(Y_val, rl.predict(V_val), average="macro")
    print(f"C={C:<5} F1 macro na validação: {f1v:.4f}")
    if melhor is None or f1v > melhor[0]:
        melhor = (f1v, C, rl)
f1_val_cong, C_cong, rl_cong = melhor
tempo_cong = time.perf_counter() - t0

def prever_cong(textos, dispositivo=DEVICE):
    return rl_cong.predict(vetores(bert_cong, textos, dispositivo=dispositivo))

pred_cong = prever_cong(T_te)
lat_cong = medir_latencia(lambda t: prever_cong([t]), T_te)
bert_cong.to("cpu"); lat_cong_cpu = medir_latencia(lambda t: prever_cong([t], torch.device("cpu")), T_te); bert_cong.to(DEVICE)

RES["BERTimbau congelado + RL"] = {**metricas(Y_te, pred_cong), "f1_val": f1_val_cong,
                                   "param_total": n_parametros(bert_cong) + rl_cong.coef_.size + rl_cong.intercept_.size,
                                   "param_treinaveis": rl_cong.coef_.size + rl_cong.intercept_.size,
                                   "tempo_treino_s": tempo_cong, "inferencia_ms": lat_cong, "inferencia_cpu_ms": lat_cong_cpu}
print(f"C escolhido: {C_cong}")
pd.Series(RES["BERTimbau congelado + RL"]).round(4)

## 4. BERTimbau ajustado (*fine-tuning*)

Agora todos os parâmetros do BERTimbau são ajustados, com uma camada de classificação por cima. Hiperparâmetros usuais para BERT (Devlin et al., 2019): taxa de aprendizado 2e-5 com aquecimento de 10% e queda linear, lote de 32, 3 épocas. A perda tem **pesos por classe**, como na etapa 4, e o modelo de cada semente é o da época com o melhor F1 macro na validação. Usamos precisão mista (fp16) na GPU, o que quase dobra a velocidade.

Como na etapa 4, são **3 sementes** (42, 43, 44) e reportamos média e desvio padrão.

### 4.1 Previsão de tempo
Antes do treino completo, a célula abaixo roda alguns passos, mede a velocidade e estima o tempo total.

In [ ]:
HP_FT = dict(lr=2e-5, batch=32, epocas=3, aquecimento=0.1, decaimento=0.01)
SEEDS = [42, 43, 44]
if RAPIDO:
    HP_FT.update(batch=16, epocas=1); SEEDS = [42]

contagem = np.bincount(Y_tr, minlength=3)
PESOS = torch.tensor(len(Y_tr) / (3 * contagem), dtype=torch.float32)
print("Pesos por classe:", dict(zip(CLASSES, PESOS.numpy().round(2))))

def novo_classificador():
    return AutoModelForSequenceClassification.from_pretrained(
        MODELO_BERT, num_labels=3, id2label=dict(enumerate(CLASSES)), label2id=IDX).to(DEVICE)

def passo_treino(modelo, otim, agenda, escalador, textos, y, perda_fn):
    enc = tok_bert(textos, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16, enabled=GPU):
        perda = perda_fn(modelo(**enc).logits.float(), torch.as_tensor(y, device=DEVICE))
    otim.zero_grad(set_to_none=True)
    escalador.scale(perda).backward()
    escalador.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    escalador.step(otim); escalador.update(); agenda.step()
    return perda.item()

def montar_otimizador(modelo, passos_totais):
    sem_decaimento = ["bias", "LayerNorm.weight"]
    grupos = [{"params": [p for n, p in modelo.named_parameters() if not any(s in n for s in sem_decaimento)],
               "weight_decay": HP_FT["decaimento"]},
              {"params": [p for n, p in modelo.named_parameters() if any(s in n for s in sem_decaimento)],
               "weight_decay": 0.0}]
    otim = torch.optim.AdamW(grupos, lr=HP_FT["lr"])
    agenda = get_linear_schedule_with_warmup(otim, int(HP_FT["aquecimento"] * passos_totais), passos_totais)
    return otim, agenda, torch.amp.GradScaler("cuda", enabled=GPU)

# --- medição: alguns passos de treino + uma passada na validação ---
fixar_semente(SEED)
modelo = novo_classificador().train()
perda_fn = torch.nn.CrossEntropyLoss(weight=PESOS.to(DEVICE))
passos_por_epoca = int(np.ceil(len(T_tr) / HP_FT["batch"]))
otim, agenda, escalador = montar_otimizador(modelo, passos_por_epoca)
N_MEDIR = 5 if RAPIDO else 30
ordem = np.random.permutation(len(T_tr))
for i in range(N_MEDIR + 3):
    if i == 3:                                   # os 3 primeiros passos são aquecimento
        if GPU: torch.cuda.synchronize()
        t0 = time.perf_counter()
    idx = ordem[(i * HP_FT["batch"]) % len(T_tr):][:HP_FT["batch"]]
    passo_treino(modelo, otim, agenda, escalador, [T_tr[j] for j in idx], Y_tr[idx], perda_fn)
if GPU: torch.cuda.synchronize()
seg_passo = (time.perf_counter() - t0) / N_MEDIR
modelo.eval(); t0 = time.perf_counter(); prever_logits(modelo, tok_bert, T_val); seg_aval = time.perf_counter() - t0
del modelo, otim; torch.cuda.empty_cache() if GPU else None

por_semente = HP_FT["epocas"] * (passos_por_epoca * seg_passo + seg_aval) + seg_aval * len(T_te) / len(T_val)
print(f"{seg_passo:.3f} s por passo | {passos_por_epoca} passos por época | {HP_FT['epocas']} épocas | {len(SEEDS)} sementes")
print(f"PREVISÃO: ~{por_semente / 60:.1f} min por semente, ~{por_semente * len(SEEDS) / 60:.0f} min no total "
      "(mais 1 a 2 min por semente para carregar o modelo e medir a inferência)")

### 4.2 Treino com 3 sementes
Cada semente também classifica os comentários do YouTube (seção 5), para que a avaliação lá tenha média e desvio padrão.

In [ ]:
def treinar_ft(seed):
    fixar_semente(seed)
    modelo = novo_classificador()
    perda_fn = torch.nn.CrossEntropyLoss(weight=PESOS.to(DEVICE))
    otim, agenda, escalador = montar_otimizador(modelo, passos_por_epoca * HP_FT["epocas"])
    melhor_f1, melhor_estado, melhor_ep, hist = -1, None, 0, []
    t0 = time.perf_counter()
    for ep in range(1, HP_FT["epocas"] + 1):
        modelo.train()
        ordem = np.random.permutation(len(T_tr))
        perdas = [passo_treino(modelo, otim, agenda, escalador, [T_tr[j] for j in idx], Y_tr[idx], perda_fn)
                  for idx in lotes(ordem, HP_FT["batch"])]
        modelo.eval()
        f1v = f1_score(Y_val, prever_logits(modelo, tok_bert, T_val).argmax(1), average="macro")
        hist.append({"epoca": ep, "perda_treino": np.mean(perdas), "f1_val": f1v})
        print(f"  semente {seed} | época {ep} | perda {np.mean(perdas):.4f} | F1 val {f1v:.4f}")
        if f1v > melhor_f1:
            melhor_f1, melhor_ep = f1v, ep
            melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    tempo = time.perf_counter() - t0
    modelo.load_state_dict(melhor_estado); modelo.eval()

    prever = lambda textos, d=DEVICE: prever_logits(modelo, tok_bert, textos, dispositivo=d).argmax(1)
    pred = prever(T_te)
    lat = medir_latencia(lambda t: prever([t]), T_te)
    modelo.to("cpu"); lat_cpu = medir_latencia(lambda t: prever([t], torch.device("cpu")), T_te); modelo.to(DEVICE)
    pred_yt = prever(YT["texto"].tolist()) if YT is not None else None

    res = {**metricas(Y_te, pred), "f1_val": melhor_f1, "melhor_epoca": melhor_ep,
           "param_total": n_parametros(modelo), "param_treinaveis": n_parametros(modelo),
           "tempo_treino_s": tempo, "inferencia_ms": lat, "inferencia_cpu_ms": lat_cpu}
    del modelo, otim; torch.cuda.empty_cache() if GPU else None
    return res, pred, pred_yt, pd.DataFrame(hist)

EXEC_FT, PRED_FT, PRED_YT_FT, HIST_FT = [], {}, {}, {}
for s in SEEDS:
    res, PRED_FT[s], PRED_YT_FT[s], HIST_FT[s] = treinar_ft(s)
    EXEC_FT.append({"configuração": "BERTimbau ajustado", "seed": s, **res})
    print(f"semente {s}: F1 macro no teste {res['f1_macro']:.4f} | {res['tempo_treino_s'] / 60:.1f} min")

exec_ft = pd.DataFrame(EXEC_FT)
num = exec_ft.drop(columns=["configuração", "seed"])
RES["BERTimbau ajustado"] = {**num.mean().to_dict(), **{f"{c}_dp": v for c, v in num.std().to_dict().items()}}
exec_ft.round(4)

## 5. Resultados

### 5.1 RePro: comparação com as etapas 2 e 4
As linhas do TF-IDF (etapa 2) e da melhor rede da etapa 4 (BiGRU + NILC GloVe ajustado) vêm de `reports/04_resultados_rede_neural.csv`. Para modelos com 3 sementes, os valores são médias (desvio padrão nas colunas `_dp`).

In [ ]:
res5 = pd.DataFrame(RES).T.rename_axis("configuração").reset_index()
anteriores = pd.read_csv(REPORTS / "04_resultados_rede_neural.csv")
anteriores = anteriores[anteriores["configuração"].isin(["TF-IDF + RL (etapa 2)", "BiGRU + NILC GloVe (ajustado)"])]
comparacao = pd.concat([anteriores, res5], ignore_index=True)
cols = ["configuração", "acuracia", "f1_macro", "f1_macro_dp", "f1_positivo", "f1_neutro", "f1_negativo",
        "param_treinaveis", "tempo_treino_s", "inferencia_ms", "inferencia_cpu_ms"]
comparacao[[c for c in cols if c in comparacao]].round(4)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
ordem = comparacao.sort_values("f1_macro")
eixos[0].barh(ordem["configuração"], ordem["f1_macro"], xerr=ordem.get("f1_macro_dp"), color="#4C72B0")
eixos[0].set_xlim(max(0, ordem["f1_macro"].min() - 0.1), 1); eixos[0].set_title("F1 macro no teste do RePro")
lat = comparacao["inferencia_cpu_ms"].fillna(comparacao["inferencia_ms"])
eixos[1].scatter(lat, comparacao["f1_macro"], color="#DD8452")
for x, y, n in zip(lat, comparacao["f1_macro"], comparacao["configuração"]):
    eixos[1].annotate(n, (x, y), fontsize=8, xytext=(4, 2), textcoords="offset points")
eixos[1].set_xscale("log"); eixos[1].set_xlabel("ms por texto na CPU (escala log)"); eixos[1].set_ylabel("F1 macro")
eixos[1].set_title("Custo × desempenho")
plt.tight_layout(); plt.show()

melhor_seed = max(EXEC_FT, key=lambda r: r["f1_val"])["seed"]
ConfusionMatrixDisplay(confusion_matrix(Y_te, PRED_FT[melhor_seed], labels=[0, 1, 2]), display_labels=CLASSES).plot(cmap="Blues")
plt.title(f"BERTimbau ajustado (semente {melhor_seed}) — teste do RePro"); plt.show()

### 5.2 YouTube
Avaliamos em duas partes, como planejado na nota de limitações:
1. **Sentimento** nos itens que os rotuladores concordaram que são positivos, neutros ou negativos, separados em comentários e títulos. Esta é a comparação justa: nenhum modelo viu YouTube no treino.
2. **O que cada modelo faz com o ruído:** nos comentários marcados "não é sobre o produto", que classe ele atribui. Nenhum modelo tem essa classe, então todos vão errar; a pergunta é *como*.

As predições são salvas só com o `id` (sem texto) em `reports/05_predicoes_youtube.csv`, para juntar depois com o TF-IDF e a BiGRU (que rodam localmente).

In [ ]:
if YT is None:
    print("Sem a amostra do YouTube: seção pulada.")
else:
    textos_yt = YT["texto"].tolist()
    preds_yt = {"XLM-R tweets (zero-shot)": prever_zs(textos_yt), "BERTimbau congelado + RL": prever_cong(textos_yt)}
    preds_yt.update({f"BERTimbau ajustado (semente {s})": p for s, p in PRED_YT_FT.items()})

    pred_tab = YT[["id", "tipo", "rotulo_final"]].copy()
    for nome, p in preds_yt.items():
        pred_tab[nome] = [CLASSES[i] for i in p]
    pred_tab.to_csv(SAIDA / "05_predicoes_youtube.csv", index=False)

    linhas = []
    for tipo in ["comentário", "título"]:
        sub = pred_tab[(pred_tab["tipo"] == tipo) & pred_tab["rotulo_final"].isin(CLASSES)]
        y = sub["rotulo_final"].map(IDX).values
        for nome in preds_yt:
            linhas.append({"conjunto": f"{tipo}s (pos/neu/neg)", "n": len(sub), "modelo": nome,
                           **metricas(y, sub[nome].map(IDX).values)})
    met_yt = pd.DataFrame(linhas)
    # agrega as sementes do BERTimbau ajustado em média ± dp
    met_yt["modelo_base"] = met_yt["modelo"].str.replace(r" \(semente \d+\)", "", regex=True)
    res_yt = met_yt.groupby(["conjunto", "n", "modelo_base"], sort=False).agg(
        **{m: (m, "mean") for m in ["acuracia", "f1_macro", "f1_positivo", "f1_neutro", "f1_negativo"]},
        f1_macro_dp=("f1_macro", "std")).reset_index().rename(columns={"modelo_base": "modelo"})
    res_yt.to_csv(SAIDA / "05_resultados_youtube.csv", index=False)
    display(res_yt.round(4))

In [ ]:
if YT is not None:
    ruido = pred_tab[pred_tab["rotulo_final"] == "nao_sobre_produto"]
    dist = pd.DataFrame({nome: ruido[nome].value_counts(normalize=True) for nome in preds_yt}).T.reindex(columns=CLASSES).fillna(0)
    print(f"Comentários 'não é sobre o produto' (n={len(ruido)}): classe atribuída por cada modelo")
    display(dist.round(3))

## 6. Arquivos de resultado

Salva as tabelas em `reports/` e, no Colab, baixa tudo num `.zip`. Depois baixe também o notebook (**Arquivo → Fazer download → Baixar o .ipynb**) e salve por cima de `src-pln/notebooks/05_transformers.ipynb`, para que as saídas apareçam no GitHub.

In [ ]:
res5.to_csv(SAIDA / "05_resultados_transformers.csv", index=False)
exec_ft.to_csv(SAIDA / "05_execucoes_por_semente.csv", index=False)
pd.concat([h.assign(seed=s) for s, h in HIST_FT.items()]).to_csv(SAIDA / "05_historico_finetuning.csv", index=False)

arquivos = sorted(SAIDA.glob("05_*.csv"))
print("Gerados:", [a.name for a in arquivos])
if NO_COLAB:
    with zipfile.ZipFile("/content/resultados_etapa5.zip", "w") as z:
        for a in arquivos:
            z.write(a, a.name)
    from google.colab import files
    files.download("/content/resultados_etapa5.zip")

## 7. Leitura dos resultados

*(A preencher depois da execução no Colab.)*